### Traveling Salesman Problem (TSP)

The Traveling Salesman Problem is one of the most important problems in operations research, as well as in mathematics and computer science. Given a graph, the goal is to find the Hamiltonian tour, that is, the route of minimum total distance that visits every node exactly once and returns to the starting point.

Let $N$ be the set of cities, and let $c_{ij}$ denote the cost of traveling from city $i$ to city $j$. We define a binary decision variable $x_{ij}$ equal to 1 if the tour goes directly from city $i$ to city $j$, for $i \neq j$, and 0 otherwise. The MTZ (Miller-Tucker-Zemlin) formulation introduces an auxiliary variable $u_i$ in order to add subtour elimination constraints.

The formulation is as follows:

$$
\begin{aligned}
\min \quad & \sum_{i \in N} \sum_{\substack{j \in N \\ j \ne i}} c_{ij} x_{ij} \\
\text{s.t.} \quad
& \sum_{\substack{i \in N \\ i \ne j}} x_{ij} = 1 && \forall j \in N \\
& \sum_{\substack{j \in N \\ j \ne i}} x_{ij} = 1 && \forall i \in N \\
& u_i - u_j + |N| x_{ij} \leq |N| - 1 && \forall i \ne j,\; i,j \in N \setminus \{0\} \\
& x_{ij} \in \{0,1\} && \forall i,j \in N \\
& u_i \in \mathbb{R} && \forall i \in N
\end{aligned}
$$


In [ ]:
%pip install -q amplpy numpy matplotlib pandas networkx folium
from amplpy import AMPL, ampl_notebook
import numpy as np

# HiGHS is the default. Gurobi requires an AMPL-compatible license.
SOLVER = "highs"  # or "gurobi"
LICENSE_UUID = "default"  # Colab Community Edition; use your UUID locally
runtime = ampl_notebook(modules=[SOLVER], license_uuid=LICENSE_UUID)

def new_ampl():
    return AMPL()

def solve_checked(model):
    model.solve(solver=SOLVER)
    if model.solve_result != "solved":
        raise RuntimeError(f"No proven optimal solution: {model.solve_result}. "
                           "Inspect the solver log before extracting values.")

def values(model, name):
    # Numeric dictionaries keep plotting independent of the solver API.
    return model.var[name].get_values().to_dict()

import itertools


In [ ]:
# Data
# Example with 5 cities
cities = {0: (0, 0), 1: (1, 5), 2: (2, 3), 3: (5, 2), 4: (6, 6)}

# Compute distance matrix
dist = {(i, j): round(((cities[i][0] - cities[j][0]) ** 2 +
                       (cities[i][1] - cities[j][1]) ** 2) ** 0.5)
        for i, j in itertools.permutations(cities, 2)}

In [ ]:
m = new_ampl()
m.eval(r"""
set N;
set A within N cross N;
param c {A} >= 0;
var x {A} binary;
var u {N} >= 0;
minimize Total_Cost: sum {(i,j) in A} c[i,j]*x[i,j];
subject to Outgoing {i in N}: sum {(i,j) in A} x[i,j] = 1;
subject to Incoming {j in N}: sum {(i,j) in A} x[i,j] = 1;
subject to MTZ {(i,j) in A: i!=0 and j!=0}:
    u[i]-u[j] + card(N)*x[i,j] <= card(N)-1;
""")
m.set["N"] = list(cities)
m.set["A"] = list(dist)
m.param["c"] = dist

solve_checked(m)
x = values(m, "x")
u = values(m, "u")
solution = [(i,j) for (i,j),value in x.items() if value > 0.5]
print("Optimal tour:", solution)


In [ ]:
import matplotlib.pyplot as plt
import random

# Extract the optimal route
solution = [(i, j) for i, j in x.keys() if x[i, j] > 0.5]
n = len(cities)

# Function to plot the route
def plot_route(route, title):
    plt.figure(figsize=(6, 6))
    for i in cities:
        plt.plot(*cities[i], 'bo')
        plt.text(cities[i][0], cities[i][1] + 0.2, f"{i}", ha='center')
    for i, j in route:
        plt.plot([cities[i][0], cities[j][0]], [cities[i][1], cities[j][1]], 'k-')
    plt.title(title)
    plt.grid(True)
    plt.show()

plot_route(solution, "Optimal route")

In [ ]:
for j in cities:
    print(f"{j}:", u[j])

In [ ]:
# Seed for reproducibility
random.seed(42)

# Generate cities with random coordinates
cities = {i: (random.randint(0, 100), random.randint(0, 100)) for i in range(44)}
n = len(cities)

# Compute Euclidean distance matrix
dist = {
    (i, j): round(((cities[i][0] - cities[j][0]) ** 2 + (cities[i][1] - cities[j][1]) ** 2) ** 0.5)
    for i, j in itertools.permutations(cities, 2)
}

In [ ]:
m = new_ampl()
m.eval(r"""
set N;
set A within N cross N;
param c {A} >= 0;
var x {A} binary;
var u {N} >= 0;
minimize Total_Cost: sum {(i,j) in A} c[i,j]*x[i,j];
subject to Outgoing {i in N}: sum {(i,j) in A} x[i,j] = 1;
subject to Incoming {j in N}: sum {(i,j) in A} x[i,j] = 1;
subject to MTZ {(i,j) in A: i!=0 and j!=0}:
    u[i]-u[j] + card(N)*x[i,j] <= card(N)-1;
""")
m.set["N"] = list(cities)
m.set["A"] = list(dist)
m.param["c"] = dist

solve_checked(m)
x = values(m, "x")
u = values(m, "u")
solution = [(i,j) for (i,j),value in x.items() if value > 0.5]
print("Optimal tour:", solution)


In [ ]:
import matplotlib.pyplot as plt
import random

# Extract the optimal route
solution = [(i, j) for i, j in x.keys() if x[i, j] > 0.5]
n = len(cities)

# Function to plot the route
def plot_route(route, title):
    plt.figure(figsize=(6, 6))
    for i in cities:
        plt.plot(*cities[i], 'bo')
        plt.text(cities[i][0], cities[i][1] + 0.2, f"{i}", ha='center')
    for i, j in route:
        plt.plot([cities[i][0], cities[j][0]], [cities[i][1], cities[j][1]], 'k-')
    plt.title(title)
    plt.grid(True)
    plt.show()

plot_route(solution, "Optimal route")